# Notebook 4.3: Preparing Features for Machine Learning

**Companion to Chapter 4: Implementing Data Pre-processing in Python**  
*Machine Learning with Python: Principles and Practical Techniques*

> **Estimated time:** 45–55 minutes  
> **Level:** Beginner to intermediate  
> **Environment:** Google Colab or Jupyter Notebook

---

## Related chapter ideas

This notebook develops data transformation for machine learning: separating features and targets, splitting data, imputing missing values, encoding categories, scaling numerical features, and combining the operations in a leakage-safe pipeline.

## Learning objectives

By the end of this notebook, you will be able to:

1. distinguish input features from a target variable;
2. create stratified training and test sets;
3. explain normalization and standardization;
4. encode categorical features with one-hot encoding;
5. apply different transformations to numerical and categorical columns;
6. construct a `ColumnTransformer` and a Scikit-learn `Pipeline`; and
7. prevent data leakage by fitting preprocessing only on training data.


## What will you build?

You will create a reusable preprocessing-and-classification pipeline for predicting whether a student passed. The model is included only to verify that the transformed features work correctly; the main focus is the preprocessing workflow.

The final workflow will follow this order:

**Raw features → Train/test split → Imputation → Scaling/encoding → Model**

> **Key principle:** The test set must represent unseen data. Any step that learns values—such as a median, mean, scale, or category list—must be fitted without looking at the test set.


## 1. Import the libraries


In [ ]:
from io import StringIO

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", None)
pd.set_option("display.precision", 3)

print("Pandas version:", pd.__version__)


## 2. Load and minimally prepare the dataset


In [ ]:
student_csv = """student_id,study_hours,attendance_pct,previous_score,learning_mode,programming_experience,assignments_submitted,final_score,passed
S001,5.5,92,78,In-person,Beginner,9,84,Yes
S002,3.0,75,65,Online,No prior experience,7,68,Yes
S003,1.5,61,58,Hybrid,No prior experience,5,55,No
S004,6.0,95,88,In-person,Intermediate,10,91,Yes
S005,2.0,70,62,Online,Beginner,6,63,Yes
S006,4.5,85,74,Hybrid,Beginner,8,79,Yes
S007,1.0,55,51,Online,No prior experience,4,48,No
S008,7.0,98,91,In-person,Advanced,10,95,Yes
S009,3.5,82,69,Hybrid,Beginner,8,73,Yes
S010,2.5,67,60,Online,No prior experience,6,59,No
S011,5.0,90,81,In-person,Intermediate,9,86,Yes
S012,4.0,88,76,Hybrid,Beginner,8,80,Yes
S013,2.0,,57,Online,No prior experience,5,54,No
S014,6.5,96,89,In-person,Advanced,10,93,Yes
S015,3.0,78,,Hybrid,Beginner,7,70,Yes
S016,1.5,63,55,Online,No prior experience,4,52,No
S017,5.5,91,83,In-person,Intermediate,9,88,Yes
S018,4.0,84,72,hybrid,Beginner,8,77,Yes
S019,2.5,72,64,Online,Beginner,6,65,Yes
S020,6.0,94,86,In-person,Advanced,10,90,Yes
S021,3.5,80,70,Hybrid,Beginner,7,72,Yes
S022,1.0,58,49,Online,No prior experience,3,45,No
S023,4.5,87,75,In-person,Intermediate,9,82,Yes
S024,2.0,69,59,Online,No prior experience,5,57,No
S025,5.0,89,80,Hybrid,Intermediate,9,85,Yes
S026,3.0,76,67,Online,Beginner,7,69,Yes
S027,6.5,97,90,In-person,Advanced,10,94,Yes
S028,1.5,60,53,Online,No prior experience,4,50,No
S029,4.0,83,73,Hybrid,Beginner,8,78,Yes
S030,2.5,74,63,Online,Beginner,6,64,Yes
S030,2.5,74,63,Online,Beginner,6,64,Yes
"""

students = pd.read_csv(StringIO(student_csv))
print("Dataset loaded successfully.")


In [ ]:
# Remove the known exact duplicate and standardize the known label inconsistency.
# Missing numerical values remain; the pipeline will learn how to impute them.
model_data = students.drop_duplicates().reset_index(drop=True).copy()
model_data["learning_mode"] = model_data["learning_mode"].str.strip().str.title()

print("Modeling table shape:", model_data.shape)
print("Missing values:")
display(model_data.isna().sum().to_frame("count"))


### Why not reuse globally imputed values from Notebook 4.2?

Notebook 4.2 created a clean exploratory table. For predictive modeling, however, imputation statistics must be learned from the training set only. We therefore keep the missing numerical values and let the pipeline handle them after the split.


## 3. Define the prediction task


In [ ]:
target_column = "passed"

numeric_features = [
    "study_hours",
    "attendance_pct",
    "previous_score",
    "assignments_submitted",
]

categorical_features = [
    "learning_mode",
    "programming_experience",
]

feature_columns = numeric_features + categorical_features

X = model_data.loc[:, feature_columns].copy()
y = model_data[target_column].map({"No": 0, "Yes": 1})

print("Feature matrix X:", X.shape)
print("Target vector y:", y.shape)
display(X.head())
display(y.value_counts().rename(index={0: "No", 1: "Yes"}).to_frame("count"))


### Think like an ML practitioner

Two columns are intentionally excluded:

- `student_id` is an identifier rather than a meaningful learning measure.
- `final_score` overlaps directly with the outcome `passed`. Including it would create **target leakage** and produce an unrealistically easy prediction problem.


## 4. Split before fitting preprocessing


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y,
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

split_balance = pd.DataFrame({
    "full": y.value_counts(normalize=True),
    "train": y_train.value_counts(normalize=True),
    "test": y_test.value_counts(normalize=True),
}).rename(index={0: "No", 1: "Yes"})

display(split_balance)


`stratify=y` approximately preserves the class proportions in both subsets. `random_state=42` makes the split reproducible.

> The test set is now placed behind a conceptual wall. We may transform it later using parameters learned from the training set, but we must not use it to choose those parameters.


## 5. Understand standardization and normalization


### Standardization

Standardization centers a feature around zero and scales it using its standard deviation:

$$z = \frac{x-\mu}{\sigma}$$

It commonly produces values with mean near 0 and standard deviation near 1. Values are not restricted to a fixed interval.

### Min-max normalization

Min-max normalization rescales a feature to a chosen range, usually 0 to 1:

$$x' = \frac{x-x_{\min}}{x_{\max}-x_{\min}}$$

It preserves relative spacing but can be strongly affected by extreme values.


In [ ]:
# Use training data only for this demonstration.
hours_train = X_train[["study_hours"]]

standard_demo = StandardScaler().fit_transform(hours_train)
minmax_demo = MinMaxScaler().fit_transform(hours_train)

scaling_demo = pd.DataFrame({
    "original": hours_train["study_hours"].to_numpy(),
    "standardized": standard_demo.ravel(),
    "normalized_0_1": minmax_demo.ravel(),
}, index=hours_train.index).sort_values("original")

display(scaling_demo.head(10))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))

for axis, column, title, color in zip(
    axes,
    ["original", "standardized", "normalized_0_1"],
    ["Original", "Standardized", "Min-max normalized"],
    ["#4C78A8", "#F58518", "#54A24B"],
):
    axis.hist(scaling_demo[column], bins=7, color=color, edgecolor="black")
    axis.set_title(title)
    axis.set_xlabel("Value")
    axis.set_ylabel("Frequency")

plt.tight_layout()
plt.show()


Scaling changes units and range, not the ordering of observations. Standardization is often suitable for linear models, distance-based methods, and principal component analysis. Min-max normalization is useful when a bounded range is desired. The best choice depends on the algorithm, distribution, and problem context.


## 6. Build the numerical preprocessing pipeline


In [ ]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

numeric_pipeline


The order matters: missing values are imputed before scaling. Each step receives the output of the previous step.


## 7. Build the categorical preprocessing pipeline


In [ ]:
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

categorical_pipeline


One-hot encoding creates a binary indicator for each learned category. `handle_unknown="ignore"` prevents failure if future data contains a category that was absent from the training set. The unknown category is represented by zeros across the learned indicators for that feature.

This setting improves robustness, but an unexpected category should still be logged and investigated.


## 8. Combine transformations with `ColumnTransformer`


In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

preprocessor


`ColumnTransformer` applies the numerical pipeline to numerical columns and the categorical pipeline to categorical columns. This is safer and easier to reproduce than transforming separate arrays manually.


## 9. Fit on training data and transform both sets


In [ ]:
X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()

X_train_ready = pd.DataFrame(
    X_train_transformed,
    columns=feature_names,
    index=X_train.index,
)
X_test_ready = pd.DataFrame(
    X_test_transformed,
    columns=feature_names,
    index=X_test.index,
)

print("Training features before:", X_train.shape)
print("Training features after:", X_train_ready.shape)
display(X_train_ready.head())


In [ ]:
print("Transformed feature names:")
for name in feature_names:
    print("-", name)


Categorical columns expanded into multiple indicator columns, so the transformed dataset contains more columns than the original feature matrix. Numerical columns retain one column each after imputation and standardization.


## 10. Verify that preprocessing learned only from training data


In [ ]:
learned_medians = preprocessor.named_transformers_["numeric"]\
    .named_steps["imputer"].statistics_

median_audit = pd.DataFrame({
    "feature": numeric_features,
    "pipeline_training_median": learned_medians,
    "direct_training_median": X_train[numeric_features].median().to_numpy(),
    "full_data_median_not_used": X[numeric_features].median().to_numpy(),
})

display(median_audit)


In [ ]:
learned_means = preprocessor.named_transformers_["numeric"]\
    .named_steps["scaler"].mean_

scale_audit = pd.DataFrame({
    "feature": numeric_features,
    "scaler_training_mean_after_imputation": learned_means,
})

display(scale_audit)


The audit shows where fitted parameters are stored. We called `fit_transform()` only on `X_train`; the test set received `transform()` only. This distinction is the heart of leakage-safe preprocessing.


## 11. Connect preprocessing and modeling in one pipeline


In [ ]:
model_pipeline = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42)),
])

model_pipeline.fit(X_train, y_train)
test_predictions = model_pipeline.predict(X_test)

print("Test accuracy:", round(accuracy_score(y_test, test_predictions), 3))
print("Confusion matrix:")
display(pd.DataFrame(
    confusion_matrix(y_test, test_predictions),
    index=["Actual No", "Actual Yes"],
    columns=["Predicted No", "Predicted Yes"],
))


The pipeline accepts the original mixed-type DataFrame and applies every learned transformation before prediction. During cross-validation or deployment, this packaging prevents preprocessing steps from becoming disconnected from the model.

> **Small-data caution:** This synthetic dataset is designed for learning, not for drawing reliable conclusions about students. A single accuracy value from eight test records is unstable and should not be treated as evidence of real-world performance.


## 12. Test the pipeline on a new record


In [ ]:
new_student = pd.DataFrame([{
    "study_hours": 3.5,
    "attendance_pct": np.nan,
    "previous_score": 71,
    "assignments_submitted": 8,
    "learning_mode": "Hybrid",
    "programming_experience": "Beginner",
}])

new_prediction = model_pipeline.predict(new_student)[0]
new_probability = model_pipeline.predict_proba(new_student)[0, 1]

print("Predicted class:", "Yes" if new_prediction == 1 else "No")
print("Illustrative probability of passing:", round(new_probability, 3))


The pipeline handled the missing attendance value automatically. In real use, a prediction should support—not replace—human judgment. A student should not be denied opportunities or labeled based on a small, unvalidated model.


## 13. Normalization alternative


In [ ]:
# Swap StandardScaler for MinMaxScaler while keeping the rest of the workflow unchanged.
minmax_numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", MinMaxScaler()),
])

minmax_preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", minmax_numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features),
    ],
    verbose_feature_names_out=False,
)

X_train_minmax = minmax_preprocessor.fit_transform(X_train)
numeric_part = X_train_minmax[:, :len(numeric_features)]

print("Minimum of each normalized numeric feature:", numeric_part.min(axis=0))
print("Maximum of each normalized numeric feature:", numeric_part.max(axis=0))


The training-set numeric features now fall between 0 and 1. A future test value can fall outside this interval if it lies beyond the training minimum or maximum; the scaler must not be refitted on that test value.


## 14. Guided practice


Complete these tasks:

1. Identify the training-set median learned for `attendance_pct`.
2. List the categories learned for `learning_mode`.
3. Transform `X_test` and confirm that it contains no missing values.
4. Explain why `preprocessor.fit_transform(X_test)` would be incorrect.


In [ ]:
# Write your solution here.


<details>
<summary><strong>Open the suggested solution</strong></summary>

```python
# 1. Training median for attendance_pct
attendance_position = numeric_features.index("attendance_pct")
print(learned_medians[attendance_position])

# 2. Learned learning-mode categories
encoder = preprocessor.named_transformers_["categorical"].named_steps["encoder"]
learning_mode_position = categorical_features.index("learning_mode")
print(encoder.categories_[learning_mode_position])

# 3. Check transformed test data
test_array = preprocessor.transform(X_test)
print(np.isnan(test_array).sum())

# 4. fit_transform on X_test would learn test-set parameters and leak information.
```

</details>


## 15. Challenge: Add a new feature


Suppose the dataset gains a categorical feature named `support_level` with values `Low`, `Medium`, and `High`.

1. Add a synthetic `support_level` column to `X`.
2. Include it in `categorical_features`.
3. Repeat the split and rebuild the pipeline.
4. Inspect the new feature names.
5. Explain why the split must be repeated from a clearly defined feature table rather than modifying only one subset.

**Responsible-use question:** Could `support_level` reflect unequal access to institutional resources? How would that affect interpretation of the model?


## 16. Common mistakes to avoid


| Mistake | Why it is a problem | Better practice |
|---|---|---|
| Scaling before splitting | Test information affects training parameters | Split first; fit preprocessing on training data |
| Encoding train and test separately | Columns may not align | Fit one encoder on training data and transform both |
| Replacing missing values with zero automatically | Zero may have a real, different meaning | Choose a justified imputation strategy |
| Including `final_score` to predict `passed` | Creates target leakage | Use only information available at prediction time |
| Keeping student ID as a feature | Encourages memorization without meaning | Exclude identifiers unless strongly justified |
| Evaluating only accuracy | Can hide class-specific errors | Examine confusion matrix and other metrics |


## 17. Reflection


1. How do standardization and min-max normalization differ?
2. Why must preprocessing be fitted after the train/test split?
3. What does one-hot encoding do to a categorical column?
4. Why is `handle_unknown="ignore"` useful, and what risk remains?
5. How does a pipeline improve reproducibility?
6. Which feature in this dataset requires the most careful ethical interpretation, and why?


## 18. Key takeaways


- Define features and targets using information that would genuinely be available at prediction time.
- Split the data before fitting imputation, scaling, or encoding.
- Standardization centers and rescales features; min-max normalization maps training values to a bounded range.
- One-hot encoding converts categories into numerical indicators.
- `ColumnTransformer` applies appropriate operations to different column groups.
- A `Pipeline` keeps preprocessing and modeling together and reduces leakage risk.
- A technically valid pipeline is not automatically a valid decision system; data quality, fairness, privacy, and human oversight remain essential.

### Looking ahead

In **Notebook 4.4: Reducing Data with Principal Component Analysis**, you will use standardized numerical features to create principal components, visualize explained variance, and interpret the benefits and limitations of dimensionality reduction.
